# Klasifikace růží

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Připojení Google Drive

In [ ]:
# Importy a konfigurace
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Cesty k datasetu a hyperparametry
BASE_PATH = Path('/content/drive/MyDrive/dataset_ruze')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_DIR = '/content/drive/MyDrive/ruze_model'
os.makedirs(MODEL_DIR, exist_ok=True)


## 2. Příprava dat

In [ ]:
# Sestavení DataFrame z adresářové struktury a příprava generátorů
classes = sorted([d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))])
rows = []
for cls in classes:
    cdir = os.path.join(BASE_PATH, cls)
    for fname in os.listdir(cdir):
        fl = fname.lower()
        if fl.endswith('.jpg') or fl.endswith('.jpeg') or fl.endswith('.png'):
            rows.append({'filepath': str(Path(cdir) / fname), 'label': cls})
df = pd.DataFrame(rows)
df = df[df['filepath'].apply(os.path.exists)].reset_index(drop=True)
# Stratifikovaný split: Train/Val/Test
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.15, stratify=train_df['label'], random_state=42)
# Augmentace pro trénink, pouze rescale pro val/test
train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, brightness_range=[0.7, 1.3], zoom_range=0.1)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)
def make_flow(gen, dataframe, shuffle=True):
    return gen.flow_from_dataframe(
        dataframe,
        x_col='filepath',
        y_col='label',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=shuffle,
    )
train_flow = make_flow(train_gen, train_df)
val_flow   = make_flow(val_gen,   val_df,  shuffle=False)
test_flow  = make_flow(test_gen,  test_df, shuffle=False)
CLASS_NAMES = list(train_flow.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)


## 3. Model a trénink

In [ ]:
# CNN architektura a trénink s EarlyStopping/ReduceLROnPlateau/ModelCheckpoint
def build_model(num_classes, img_size):
    model = keras.Sequential([
        keras.Input(shape=(*img_size, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model
model = build_model(NUM_CLASSES, IMG_SIZE)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
checkpoint_path = os.path.join(MODEL_DIR, 'best_model.keras')
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1),
]
history = model.fit(train_flow, validation_data=val_flow, epochs=EPOCHS, callbacks=callbacks)


## 4. Vyhodnocení a uložení

In [ ]:
# Vyhodnocení na testu a uložení modelu + tříd na Drive
test_loss, test_acc = model.evaluate(test_flow, verbose=0)
print(test_acc)
test_flow.reset()
y_pred = np.argmax(model.predict(test_flow, verbose=0), axis=1)
y_true = test_flow.classes
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
save_path = os.path.join(MODEL_DIR, 'ruze_model.keras')
model.save(save_path)
with open(os.path.join(MODEL_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f)
print('Model ulozen do: ' + save_path)
